# 05 — Multimodal MoE: ảnh + metadata (MILK10k)

Mỗi lesion được biểu diễn bằng các **token** (d = 256): ảnh **clinical**, ảnh **dermoscopy** (theo lựa chọn nhánh ảnh),
**demographics + vị trí** (20 chiều) và **14 điểm MONET**. Metadata luôn được dùng. Encoder ảnh dùng chung cho 2 ảnh.

| Hướng | Cách trộn token | Head |
|---|---|---|
| **A** | Gate theo nguồn: mỗi nguồn 1 expert, gate theo từng lesion trộn đặc trưng | 1 head |
| **B** | Transformer + sparse MoE: attention giữa các token, mỗi token qua top-2 / 4 expert dùng chung | 1 head |
| **C** | Như B | 3 head long-tail (logit-adjusted, τ = 0 / 0.5 / 1) |
| **D** | Như A | 3 head long-tail |
| **E** | Không MoE: nối 4 token rồi qua MLP (trộn thông thường) — đối chứng cho C, D | 3 head long-tail |

Chung cho cả 5 hướng: head phụ cho từng nguồn (deep supervision), modality dropout, train end-to-end.

**Chọn để chạy:** hướng × loss × backbone × nhánh ảnh — notebook train **mọi tổ hợp**.
- **Loss** = overlay trong `configs/imbalance/`. Hướng **C, D, E** đã có loss riêng → chỉ dùng overlay `sampler_*`
  và thêm 1 run không overlay (tag `lt`).
- Tên run: `<backbone>__<hướng>__<loss>__<nhánh ảnh>`, vd `vit_base__B__ce_sqrtinv__clin+derm`.
- `oof.csv` có thêm cột giải thích: `gate_*` (A, D: mức dựa vào từng nguồn) hoặc `expert_*` (B, C: expert top-1 của token).
- So sánh gợi ý: **A↔D**, **B↔C** (chỉ khác head) và **E↔C, D** (cùng 3 head long-tail, có MoE hay không).

Mặc định: ViT-B (tốt nhất, nhanh nhất ở stage 2), CE + √inv và sampler q=0.5 (2 loss tốt nhất), 3 nhánh ảnh, 5 hướng
→ **30 run**, ~7 giờ trên 2 GPU T4.

### Cách chạy trên Kaggle
1. **Settings → Accelerator: GPU T4 x2**, **Internet: On**.
2. Ô *Cấu hình* đã có sẵn link repo và link dữ liệu; chỉ cần sửa `RESTORE` khi chạy tiếp.
3. **Save Version → Save & Run All (Commit)**: notebook chạy nền tối đa 12 giờ, không cần mở trình duyệt.
   Toàn bộ `outputs/` (log, bảng, file nộp) được lưu vào output của version.
4. **Chưa chạy hết lưới** (hết 12 giờ): Add Input → output của version vừa chạy, điền vào `RESTORE`,
   rồi Save Version lần nữa. Các run đã xong được bỏ qua, run dở dang được chạy lại.

Hai GPU được dùng song song: mỗi GPU chạy một run độc lập (`scripts/run_grid.py`). Checkpoint `.pt` bị xoá
sau khi sinh file nộp để không vượt giới hạn 20GB output; `test_probs.csv` được giữ lại để ensemble.

**So sánh với chỉ ảnh:** gắn output của notebook 02 làm Input và điền vào `REFERENCE` → bảng kết quả có thêm cột
*Chỉ ảnh (stage 2)* cùng backbone, cùng loss.

In [ ]:
# ===================== CẤU HÌNH =====================
REPO_URL = "https://github.com/trong5nhan6/fussion.git"
BRANCH = "main"
DRIVE_URL = "https://drive.google.com/file/d/1lM1nB_4pq3Vg-GwlaTMQjtus_bpbMhGX/view?usp=drive_link"  # zip dữ liệu MILK10k
DATA_SOURCE = None    # hoặc đường dẫn Kaggle Dataset, vd "/kaggle/input/milk10k" (khi đó không tải từ Drive)

# Chạy tiếp / dùng lại kết quả: gắn output của notebook trước làm Input (Add Input -> Your Work -> Notebooks),
# rồi liệt kê thư mục outputs của nó, vd "/kaggle/input/<ten-notebook>/fussion/outputs"
RESTORE = []
TIME_BUDGET_H = 11.0

# ===================== CHỌN TỔ HỢP ĐỂ TRAIN =====================
ARCHS     = ["A", "B", "C", "D", "E"]      # A gate theo nguồn | B transformer MoE | C = B + long-tail
                                           #   D = A + long-tail | E = nối 4 token + long-tail (không MoE)
LOSSES    = ["ce_sqrt_inv", "sampler_q05"] # tên file trong configs/imbalance/: ce_plain, ce_sqrt_inv, ce_inv,
                                           #   cb_effective_num, cb_focal, logit_adjusted, sampler_q05, sampler_q1, crt
BACKBONES = ["vit_base"]                   # vit_base | swin_base | convnext_base | resnet152 | dinov2_base
VIEWS     = ["clin", "derm", "clin+derm"]  # nhánh ảnh (metadata luôn được dùng)

# Thư mục outputs của notebook 02 (chỉ để so sánh trong bảng, không train lại), vd "/kaggle/input/<nb02>/fussion/outputs"
REFERENCE = []

In [ ]:
# ===================== THAM SỐ TRAIN =====================
# Áp dụng cho MỌI run của lưới, ưu tiên cao nhất (ghi đè configs/default.yaml, configs/baselines/*, configs/env/kaggle_t4.yaml).
# None = giữ giá trị trong config. Một số tham số khác nhau theo backbone, chỉ đặt khi muốn áp dụng đồng loạt:
#   img_size: ResNet-152 320, ConvNeXt/ViT/Swin 224 (Swin-B window 7 bắt buộc 224)
#   batch_size x accum_steps: đặt 16 x 4 (batch hiệu dụng 64) cho MỌI backbone, kể cả ResNet-152
#   (giống các run stage 2 đã train; None = theo config: 16 x 2, ResNet-152 8 x 4)
#   lr: ResNet-152 3e-4, ConvNeXt-B 1e-4, ViT-B / Swin-B 5e-5
# Xem bảng "Cấu hình thực tế" ở bước Danh sách run để kiểm tra trước khi train.
PARAMS = {
    # --- Huấn luyện ---
    "train.epochs": 20,                # số epoch tối đa
    "train.patience": 5,               # dừng sớm sau N epoch Dice val không tăng
    "train.warmup_epochs": 1,          # warmup tuyến tính rồi cosine
    "train.lr": None,                  # lr của head; backbone = lr * backbone_lr_mult
    "train.backbone_lr_mult": None,
    "train.weight_decay": None,
    "train.batch_size": 16,            # số mẫu mỗi lần forward (giới hạn bởi VRAM)
    "train.accum_steps": 4,            # cộng dồn gradient -> batch hiệu dụng = 16 x 4 = 64
    "train.amp": True,                 # mixed precision fp16
    "train.monitor": "dice",           # metric chọn checkpoint: dice | auc | ap | balanced_acc
    "train.folds": [0],                # [0, 1, 2, 3, 4] khi đã chốt method và chạy đủ 5 fold
    "train.full_data": False,          # True: train trên TOÀN BỘ dữ liệu, không val, dùng epoch cuối -> đặt
                                       #   train.epochs = epoch tốt nhất đã thấy khi có val; nên đặt RUN_SUFFIX = "full"
    "seed": 42,
    # --- Dữ liệu & mô hình ---
    "data.img_size": None,
    "model.pretrained": True,
    "model.dropout": None,
    # --- Đánh giá & file nộp ---
    "predict.tta": True,               # trung bình ảnh gốc + lật ngang + lật dọc
    "predict.postprocess": "auto",     # auto: sigmoid nếu loss=bce, ngược lại top1 | top1 | softmax | sigmoid
}
# --- Tham số của từng phương pháp imbalance (chỉ có tác dụng với overlay tương ứng) ---
PARAMS.update({
    "loss.cb_beta": None,              # cb_effective_num, cb_focal (mặc định 0.999)
    "loss.focal_gamma": None,          # cb_focal (mặc định 2.0)
    "loss.la_tau": None,               # logit_adjusted (mặc định 1.0)
    "crt.epochs": None,                # crt: số epoch train lại head (mặc định 10)
    "crt.lr": None,                    # crt: lr của head (mặc định 1e-3)
})
# KHÔNG đặt loss.name / loss.class_weight / train.sampler_q / crt.enabled ở đây: chúng ghi đè cả 9 overlay.
# --- Tham số MoE (mục `moe` trong configs/default.yaml) ---
PARAMS.update({
    "moe.d_model": 256,                # kích thước token chung
    "moe.aux_weight": 0.25,            # loss phụ của head từng nguồn (deep supervision)
    "moe.modality_dropout": 0.15,      # xác suất che mỗi nguồn khi train
    "moe.fusion_lr_mult": 1.0,         # lr phần MoE mới khởi tạo = train.lr * hệ số
    # B, C: transformer + sparse MoE
    "moe.n_experts": 4,
    "moe.top_k": 2,
    "moe.n_blocks": 2,
    "moe.n_heads": 4,
    "moe.balance_alpha": 0.01,         # load-balancing loss
    # C, D, E: 3 head long-tail (logit-adjusted), τ của từng head
    "moe.lt_taus": [0.0, 0.5, 1.0],
})

# Đổi tham số so với lần chạy trước -> đặt hậu tố, vd "ep30". Run trùng tên đã xong sẽ bị BỎ QUA,
# nên không có hậu tố thì kết quả cũ được giữ nguyên thay vì train lại.
RUN_SUFFIX = ""

## 1. Cài đặt

In [ ]:
import os, glob
from IPython.display import Markdown, display

REPO_DIR = "/kaggle/working/" + REPO_URL.rstrip("/").split("/")[-1].removesuffix(".git")
DATA_DIR = "/tmp/MILK10k"               # ngoài /kaggle/working -> dữ liệu không bị lưu vào output notebook
os.environ["MILK10K_ROOT"] = DATA_DIR   # code tự đọc biến này thay cho data.root trong config

if os.path.isdir(REPO_DIR):
    !git -C {REPO_DIR} pull --ff-only
else:
    !git clone --depth 1 -b {BRANCH} {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!pip install -q gdown
!git log -1 --format="Commit: %h %s"

In [ ]:
!nvidia-smi --query-gpu=index,name,memory.total --format=csv
import torch
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available(), "| số GPU:", torch.cuda.device_count())
assert torch.cuda.device_count() >= 1, "Chưa bật GPU: Settings -> Accelerator -> GPU T4 x2"

## 2. Dữ liệu

In [ ]:
src = f"--source {DATA_SOURCE}" if DATA_SOURCE else f'--drive-url "{DRIVE_URL}"'
!python scripts/setup_data.py {src} --dest {DATA_DIR}

In [ ]:
# Kiểm tra nhanh toàn bộ pipeline (không cần GPU, ~1 phút)
!python -m pytest tests -q -p no:cacheprovider

## 3. Danh sách run, cấu hình thực tế và thời gian ước tính

In [ ]:
restore_arg = ("--restore " + " ".join(RESTORE)) if RESTORE else ""
for r in RESTORE:
    print(r, "->", "OK" if os.path.isdir(r) else "KHÔNG TỒN TẠI (kiểm tra lại đường dẫn Input)")

In [ ]:
import shlex, yaml

def to_set_args(params: dict) -> str:
    # Chuyển PARAMS thành chuỗi --set key=value cho scripts (bỏ qua giá trị None)
    items = []
    for k, v in params.items():
        if v is None:
            continue
        val = yaml.safe_dump(v, default_flow_style=True).strip().removesuffix("...").strip()
        items.append(shlex.quote(f"{k}={val}"))
    return ("--set " + " ".join(items)) if items else ""

set_arg = to_set_args(PARAMS)
suffix_arg = f"--suffix {RUN_SUFFIX}" if RUN_SUFFIX else ""
print("Override:", set_arg or "(không có)")
print("Hậu tố tên run:", RUN_SUFFIX or "(không có)")

In [ ]:
select_arg = (f"--archs {','.join(ARCHS)} --overlays {','.join(LOSSES)} "
              f"--backbones {','.join(BACKBONES)} --views {','.join(VIEWS)}")

# Ước tính: phút/run đo ở stage 2 (T4, batch 16 x 4) + ~10% cho phần MoE; 2 run song song trên 2 GPU
MIN_PER_RUN = {"vit_base": 28, "swin_base": 36, "convnext_base": 31, "resnet152": 50, "dinov2_base": 33}
samplers = [l for l in LOSSES if l.startswith("sampler")]
runs_per_view = sum(len(LOSSES) if a in "AB" else 1 + len(samplers) for a in ARCHS)
n_runs = runs_per_view * len(VIEWS) * len(BACKBONES)
hours = sum(runs_per_view * len(VIEWS) * MIN_PER_RUN[b] for b in BACKBONES) / 60 / 2
print(f"{n_runs} run, ước tính ~{hours:.1f} giờ trên 2 GPU (giới hạn mỗi phiên: {TIME_BUDGET_H} giờ)")
if hours > TIME_BUDGET_H:
    print("-> Không xong trong 1 phiên: chạy tiếp ở phiên sau bằng RESTORE, hoặc bớt lựa chọn.")
!python scripts/run_grid.py experiments/stage3_moe.yaml --env configs/env/kaggle_t4.yaml {select_arg} {set_arg} {suffix_arg} {restore_arg} --dry-run

## 4. Train

In [ ]:
!python scripts/run_grid.py experiments/stage3_moe.yaml --env configs/env/kaggle_t4.yaml {select_arg} {set_arg} {suffix_arg} {restore_arg} --gpus auto --time-budget-h {TIME_BUDGET_H}

## 5. Kết quả

In [ ]:
ref_dirs = " ".join(r for r in REFERENCE if os.path.isdir(r))
!python scripts/summarize.py --dirs outputs {ref_dirs} > /dev/null
display(Markdown(open("outputs/summary/summary.md", encoding="utf-8").read()))

## Nén kết quả để tải về

In [ ]:
import zipfile, datetime
from pathlib import Path

zip_path = Path("/kaggle/working") / f"05_multimodal_moe_outputs_{datetime.datetime.now():%Y%m%d_%H%M}.zip"
src = Path(REPO_DIR) / "outputs"
n = 0
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for f in sorted(src.rglob("*")):
        if f.is_file() and f.suffix != ".pt":          # không đóng gói checkpoint
            z.write(f, Path("outputs") / f.relative_to(src))
            n += 1
print(f"Đã nén {n} file -> {zip_path} ({zip_path.stat().st_size / 2**20:.1f} MB)")
print("Tải về: tab Output của version notebook (bên phải) -> {} -> Download".format(zip_path.name))